# Chapter 1 — What Is Reinforcement Learning?
## Concept Notebook: The RL Loop from First Principles

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/rl-for-data-scientists/blob/main/notebooks/chapter_01_what_is_rl/01_rl_intuition.ipynb)

**No GPU required. Runtime: ~10 minutes.**

---

This notebook implements the core RL loop using a simple "GridWorld" environment — the same environment discussed in Chapter 1 of the book. We go from the A/B test analogy to a working RL agent in under 100 lines of Python.

**What you'll build:**
- A minimal GridWorld environment (5×5 grid, agent finds goal)
- A random agent as baseline
- A greedy Q-table agent
- A learning curve comparison plot

In [ ]:
# No external dependencies needed for this chapter
import numpy as np
import matplotlib.pyplot as plt
import random
from collections import defaultdict

print('All imports successful. No GPU needed for Chapter 1.')

## 1.1 The Environment: GridWorld

A 5×5 grid. The agent starts at (0,0). The goal is at (4,4). Walls block some cells.
Actions: UP, DOWN, LEFT, RIGHT.
Reward: +10 at goal, -1 per step (encourages efficiency), -5 for hitting a wall.

In [ ]:
class GridWorld:
    """Simple 5x5 GridWorld with walls."""
    
    def __init__(self, size=5):
        self.size = size
        self.goal = (size-1, size-1)
        self.walls = {(1,1), (1,2), (2,2), (3,1)}  # wall positions
        self.actions = ['UP', 'DOWN', 'LEFT', 'RIGHT']
        self.action_map = {
            'UP':    (-1,  0),
            'DOWN':  ( 1,  0),
            'LEFT':  ( 0, -1),
            'RIGHT': ( 0,  1)
        }
        self.reset()
    
    def reset(self):
        self.pos = (0, 0)
        return self.pos
    
    def step(self, action):
        dr, dc = self.action_map[action]
        new_r = self.pos[0] + dr
        new_c = self.pos[1] + dc
        
        # Check boundaries and walls
        if (0 <= new_r < self.size and 
            0 <= new_c < self.size and 
            (new_r, new_c) not in self.walls):
            self.pos = (new_r, new_c)
            reward = -5  # wall penalty — invalid
        else:
            reward = -5  # stayed in place
        
        if self.pos == self.goal:
            reward = +10
            done = True
        else:
            reward += -1  # step cost
            done = False
        
        return self.pos, reward, done
    
    def visualize(self, title='GridWorld'):
        grid = np.zeros((self.size, self.size))
        for (r, c) in self.walls:
            grid[r][c] = -1
        gr, gc = self.goal
        grid[gr][gc] = 2
        pr, pc = self.pos
        grid[pr][pc] = 1
        
        fig, ax = plt.subplots(1, 1, figsize=(4, 4))
        ax.imshow(grid, cmap='RdYlGn', vmin=-1, vmax=2)
        ax.set_title(title)
        ax.set_xticks(range(self.size))
        ax.set_yticks(range(self.size))
        for r in range(self.size):
            for c in range(self.size):
                if (r, c) == self.pos:
                    ax.text(c, r, '🤖', ha='center', va='center', fontsize=14)
                elif (r, c) == self.goal:
                    ax.text(c, r, '🏆', ha='center', va='center', fontsize=14)
                elif (r, c) in self.walls:
                    ax.text(c, r, '█', ha='center', va='center', fontsize=14)
        plt.tight_layout()
        plt.show()

env = GridWorld()
env.visualize('Initial GridWorld State')

## 1.2 The RL Loop: Observe → Decide → Act → Observe

This is the core loop from Chapter 1. Let's implement it explicitly.

In [ ]:
def run_episode(env, agent_fn, max_steps=50):
    """Run one episode. agent_fn maps state -> action."""
    state = env.reset()
    total_reward = 0
    trajectory = [state]
    
    for step in range(max_steps):
        # OBSERVE current state
        # DECIDE which action to take
        action = agent_fn(state)
        # ACT and get next observation
        next_state, reward, done = env.step(action)
        
        total_reward += reward
        trajectory.append(next_state)
        state = next_state
        
        if done:
            break
    
    return total_reward, step + 1, trajectory

# Baseline: Random agent
def random_agent(state):
    return random.choice(['UP', 'DOWN', 'LEFT', 'RIGHT'])

# Run 1000 random episodes and collect rewards
random_rewards = []
for _ in range(1000):
    reward, steps, _ = run_episode(GridWorld(), random_agent)
    random_rewards.append(reward)

print(f'Random agent — mean reward: {np.mean(random_rewards):.2f} ± {np.std(random_rewards):.2f}')

## 1.3 A Learning Agent: Q-Table

The agent maintains a table Q[state][action] estimating future reward. After each step it updates the table using the TD rule from Chapter 5.

For now, just observe that the agent *improves over time* — this is the key insight of Chapter 1.

In [ ]:
class SimpleQLearner:
    def __init__(self, actions, alpha=0.3, gamma=0.9, epsilon=0.3):
        self.actions = actions
        self.alpha = alpha    # learning rate
        self.gamma = gamma    # discount factor
        self.epsilon = epsilon  # exploration rate
        self.Q = defaultdict(lambda: defaultdict(float))
    
    def choose_action(self, state):
        # Epsilon-greedy: explore randomly or exploit best known action
        if random.random() < self.epsilon:
            return random.choice(self.actions)
        q_vals = {a: self.Q[state][a] for a in self.actions}
        return max(q_vals, key=q_vals.get)
    
    def update(self, s, a, r, s_next, done):
        # TD update (Chapter 5 will explain this in depth)
        best_next = max(self.Q[s_next][a2] for a2 in self.actions)
        target = r + (0 if done else self.gamma * best_next)
        self.Q[s][a] += self.alpha * (target - self.Q[s][a])

def train_q_agent(n_episodes=2000):
    agent = SimpleQLearner(actions=['UP','DOWN','LEFT','RIGHT'])
    rewards_per_episode = []
    
    for ep in range(n_episodes):
        env = GridWorld()
        state = env.reset()
        total_reward = 0
        
        for _ in range(50):
            action = agent.choose_action(state)
            next_state, reward, done = env.step(action)
            agent.update(state, action, reward, next_state, done)
            total_reward += reward
            state = next_state
            if done:
                break
        
        rewards_per_episode.append(total_reward)
        # Decay exploration over time
        agent.epsilon = max(0.05, agent.epsilon * 0.999)
    
    return agent, rewards_per_episode

agent, rewards = train_q_agent(n_episodes=2000)
print('Training complete!')

In [ ]:
# Plot the learning curve — the key visual of Chapter 1
window = 50
smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rewards, alpha=0.2, color='steelblue', label='Episode reward')
ax.plot(range(window-1, len(rewards)), smoothed, color='steelblue', 
        linewidth=2, label=f'{window}-episode moving average')
ax.axhline(y=np.mean(random_rewards), color='red', linestyle='--', 
           label=f'Random agent baseline ({np.mean(random_rewards):.1f})')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('The Learning Curve: Q-Learning Agent on GridWorld')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nFinal 100 episodes mean reward: {np.mean(rewards[-100:]):.2f}')
print(f'Random agent mean reward: {np.mean(random_rewards):.2f}')
print(f'Improvement: {np.mean(rewards[-100:]) - np.mean(random_rewards):.2f} reward units')

## ✅ Key Takeaways from Chapter 1

1. **The RL loop** is Observe → Decide → Act → Observe. Everything in this book is built on this loop.
2. **Learning from experience** (not labels): the agent improved from `-∞` to a high reward purely by interacting with the environment.
3. **The learning curve** is the fundamental diagnostic: reward should increase over episodes. If it doesn't, something is wrong (usually the reward function or the exploration strategy).
4. **Exploration vs. exploitation**: the epsilon parameter controls this tradeoff. Chapter 9 goes deep on multi-armed bandits, which formalise this precisely.

**Next**: Chapter 2 reviews the probability and statistics prerequisites before we go deep on the mathematics.

---
*Part of the companion repository for "Reinforcement Learning for Data Scientists" — [github.com/yourusername/rl-for-data-scientists](https://github.com/yourusername/rl-for-data-scientists)*